In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.graph_builder import load_graph
from src.models import compare_with_models, compare_countries, save_model_comparison_csv

OUTPUT_DIR = PROJECT_ROOT / "output"
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

In [2]:
G_us = load_graph("US", OUTPUT_DIR / "graphs")
G_gb = load_graph("GB", OUTPUT_DIR / "graphs")

print(f"US: {G_us.number_of_nodes():,} node, {G_us.number_of_edges():,} edge")
print(f"GB: {G_gb.number_of_nodes():,} node, {G_gb.number_of_edges():,} edge")

US: 2,195 node, 69,037 edge
GB: 1,617 node, 64,840 edge


In [3]:
model_comparison_us = compare_with_models(G_us)
model_comparison_us["country"] = "US"
model_comparison_us

f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\fitting.py:528: UserWarning: No valid values for xmin found.
  warnings.warn('No valid values for xmin found.')
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\distributions.py:743: OptimizeWarning: Initial guess is not within the specified bounds
  result = scipy.optimize.minimize(fit_function,
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted parameters are very close to the edge of parameter ranges for distribution power_law; consider changing these ranges.
  warnings.warn(f'Fitted parameters are very close to the edge of parameter ranges for distribution {self.name}; consider changing these ranges.')
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\fitting.py:528: UserWarning: No valid values for xmin found.
  warnings.warn('No valid values for xmin found.')
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\distributions.py:743: Optim

,model,num_nodes,num_edges,average_clustering,average_shortest_path_length,diameter,power_law_alpha,country
0,Real,2195,69037,0.757868,2.785919,10,2.688680,US
1,Erdos-Renyi (ER),2195,69037,0.028701,2.131572,3,3.000000,US
2,Barabasi-Albert (BA),2195,67084,0.074907,2.149832,3,2.883013,US
3,Watts-Strogatz (WS),2195,70240,0.539596,2.621621,4,3.000000,US


In [4]:
model_comparison_gb = compare_with_models(G_gb)
model_comparison_gb["country"] = "GB"
model_comparison_gb

f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\fitting.py:528: UserWarning: No valid values for xmin found.
  warnings.warn('No valid values for xmin found.')
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\distributions.py:743: OptimizeWarning: Initial guess is not within the specified bounds
  result = scipy.optimize.minimize(fit_function,
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted parameters are very close to the edge of parameter ranges for distribution power_law; consider changing these ranges.
  warnings.warn(f'Fitted parameters are very close to the edge of parameter ranges for distribution {self.name}; consider changing these ranges.')
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\fitting.py:528: UserWarning: No valid values for xmin found.
  warnings.warn('No valid values for xmin found.')
f:\anaconda3\envs\youtube-network\lib\site-packages\powerlaw\distributions.py:743: Optim

,model,num_nodes,num_edges,average_clustering,average_shortest_path_length,diameter,power_law_alpha,country
0,Real,1617,64840,0.765690,2.817821,7,2.637266,GB
1,Erdos-Renyi (ER),1617,64840,0.049544,1.967937,3,3.000000,GB
2,Barabasi-Albert (BA),1617,63080,0.108698,1.995970,3,2.886235,GB
3,Watts-Strogatz (WS),1617,64680,0.544883,2.356681,3,3.000000,GB


In [5]:
model_comparison_all = pd.concat(
    [model_comparison_us, model_comparison_gb], ignore_index=True
)
save_model_comparison_csv(model_comparison_all, OUTPUT_DIR / "tables", "model_comparison.csv")
model_comparison_all

,model,num_nodes,num_edges,average_clustering,average_shortest_path_length,diameter,power_law_alpha,country
0,Real,2195,69037,0.757868,2.785919,10,2.688680,US
1,Erdos-Renyi (ER),2195,69037,0.028701,2.131572,3,3.000000,US
2,Barabasi-Albert (BA),2195,67084,0.074907,2.149832,3,2.883013,US
3,Watts-Strogatz (WS),2195,70240,0.539596,2.621621,4,3.000000,US
4,Real,1617,64840,0.765690,2.817821,7,2.637266,GB
5,Erdos-Renyi (ER),1617,64840,0.049544,1.967937,3,3.000000,GB
6,Barabasi-Albert (BA),1617,63080,0.108698,1.995970,3,2.886235,GB
7,Watts-Strogatz (WS),1617,64680,0.544883,2.356681,3,3.000000,GB


In [6]:
def load_country_metrics(country: str) -> dict:
    tables_dir = OUTPUT_DIR / "tables"
    basic_path = tables_dir / f"{country}_basic_metrics.json"
    powerlaw_path = tables_dir / f"{country}_powerlaw.json"

    if not basic_path.exists() or not powerlaw_path.exists():
        raise FileNotFoundError(
            f"Thiếu file metrics cho {country}. Hãy chạy 02_analysis.ipynb "
            f"với COUNTRY = \"{country}\" trước."
        )

    with open(basic_path, encoding="utf-8") as f:
        basic = json.load(f)
    with open(powerlaw_path, encoding="utf-8") as f:
        powerlaw_metrics = json.load(f)

    return {**basic, **powerlaw_metrics}

metrics_us = load_country_metrics("US")
metrics_gb = load_country_metrics("GB")

In [7]:
country_comparison_df = compare_countries(metrics_us, metrics_gb)
country_comparison_df

,metric,US,GB
0,alpha,2.68868,2.637266
1,average_clustering,0.757868,0.76569
2,average_shortest_path_length,2.785919,2.817821
3,density,0.028671,0.049627
4,diameter,10,7
5,favors_power_law,False,False
6,ks_distance,0.054785,0.078534
7,largest_cc_fraction,0.996355,0.974026
8,largest_cc_size,2187,1575
9,loglikelihood_ratio_vs_exponential,0.591983,-4.178815


In [8]:
country_comparison_path = OUTPUT_DIR / "tables" / "US_vs_GB_comparison.csv"
country_comparison_df.to_csv(country_comparison_path, index=False)
print(f"Đã lưu: {country_comparison_path}")

Đã lưu: F:\YOUTUBE _ TRENDING\output\tables\US_vs_GB_comparison.csv
